# 第2章 ADS-B
1090MHzのIQデータから、飛行機の位置を求めます。

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, "..")  # code/ 直下のモジュールを読み込む
DATA = Path(os.environ.get("SDR_BOOK_DATA", "../data"))  # IQ データの置き場所
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import adsb

x = adsb.load_iq(DATA / "adsb.iq")
mag = adsb.magnitude(x)
starts = adsb.find_preambles(mag)
print("preamble candidates:", len(starts))

## プリアンブルの付近の振幅

In [ ]:
s = starts[0]
plt.figure(figsize=(9, 3))
plt.plot(mag[s - 20:s + 16 + 2 * 112])
plt.xlabel("sample"); plt.ylabel("|x|"); plt.show()

## PPMの復調とCRCの検査

In [ ]:
frames = []
for s in starts:
    bits = adsb.demod_ppm(mag, s)
    if bits is None or adsb.bits_to_int(bits[0:5]) not in (17, 18) or adsb.crc24(bits) != 0:
        continue
    frames.append((s, bits))
print("CRC OK:", len(frames))
for s, bits in frames[:5]:
    print(adsb.bits_to_hex(bits), adsb.decode_frame(bits))

## 機体ごとにまとめて地図に描く

In [ ]:
aircraft = {}
for s, bits in frames:
    adsb.update_aircraft(aircraft, adsb.decode_frame(bits), s / adsb.SAMPLE_RATE)
for icao, ac in aircraft.items():
    if ac["track"]:
        print(icao, ac.get("callsign", "-"), ac.get("altitude"), ac["track"][-1])
adsb.plot_map(aircraft, "adsb_map.png")